# Score new customers as data arrives

Learn how to train a model once and keep scoring as data arrives: new customers daily and every customer weekly.
This Kumo Engine module trains on one version of a synthetic retail dataset, then refreshes the data twice with new customers and transactions and scores each refresh with the same model.
It answers two questions: does a model score customers that were absent from its training data, and how do you pick "new since last run" when customers have no created timestamp?

Use a Linux notebook prepared with the [Kumo Engine setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/linux-machine-deployment). A GPU is recommended for the FAST training preset; set `SMOKE_TEST = True` below for a quick check on a CPU-only machine.
Expect under a minute on a single L4 GPU after installation.

See [Refresh a model](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/deploy-a-model/refresh-a-model) and [Making predictions](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/making-predictions).

## Install

Install Kumo Engine in this notebook kernel.

In [1]:
%pip install kumo-engine

## Choose a workspace

Set `KUMO_LOCAL_SSD` to a local SSD mount for faster scratch storage; otherwise the system temporary directory is used.
This cell creates a new directory on each run, with separate source and scratch directories, and does not replace existing data.
The last lines print the installed Kumo Engine version and the workspace.

In [2]:
import importlib.metadata
import os
import tempfile
from pathlib import Path

from collections import namedtuple

import numpy as np
import pandas as pd
from kumoapi.model_plan import RunMode
from kumoapi.typing import Stype

import kumo.engine as kumo

workspace_root = Path(os.environ.get("KUMO_LOCAL_SSD", tempfile.gettempdir()))
workspace = Path(tempfile.mkdtemp(prefix="kumo-incremental-scoring-", dir=workspace_root))
source_dir = workspace / "source"
source_dir.mkdir()
kumo.configure(scratch_dir=workspace / "scratch")
print("Kumo Engine:", importlib.metadata.version("kumo-engine"))
print("Workspace:", workspace)

Kumo Engine: 1.0.1
Workspace: /tmp/kumo-incremental-scoring-9ippwm19


## Set the training budget

Models train with the [FAST preset](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/model-plan-guide).
Set `SMOKE_TEST = True` to cap training at two short epochs for a quick check on a CPU-only machine; its scores carry no meaning.

In [3]:
SMOKE_TEST = False

## Generate synthetic example data

Run the next cell to create three versions of synthetic customer and transaction tables.
`v1` is the training data: 500 customers and 2,500 transactions over 120 days. `v2` adds eight days of transactions and 120 new customers; `v3` adds another eight days and 70 more new customers.
`customers` has no created timestamp. In each refresh, most new customers have already transacted and some have not.
You can skim the generation code; the Kumo workflow starts below.

In [4]:
rng = np.random.default_rng(0)
start = pd.Timestamp("2024-01-01")
existing_ids = np.arange(500)
active_ids_v2 = np.arange(500, 600)
active_ids_v3 = np.arange(620, 680)


def make_transactions(count, first_day, last_day, customer_ids):
    return pd.DataFrame({
        "CUSTOMER_ID": rng.choice(customer_ids, count),
        "TS": start + pd.to_timedelta(rng.integers(first_day, last_day, count), unit="D"),
        "AMOUNT": rng.gamma(2.0, 40.0, count).round(2),
    })


def write_version(name, n_customers, transactions):
    folder = source_dir / name
    folder.mkdir()
    customer_ids = np.arange(n_customers)
    pd.DataFrame({"CUSTOMER_ID": customer_ids, "AGE": customer_ids % 60 + 20}
                 ).to_parquet(folder / "customers.parquet", index=False)
    transactions.reset_index(drop=True).rename_axis("TX_ID").reset_index(
    ).to_parquet(folder / "transactions.parquet", index=False)
    return folder


history = make_transactions(2500, 0, 120, existing_ids)
week_2 = pd.concat([
    make_transactions(120, 120, 128, existing_ids),
    make_transactions(60, 120, 128, active_ids_v2),
])
week_3 = pd.concat([
    make_transactions(130, 128, 136, np.concatenate([existing_ids, active_ids_v2])),
    make_transactions(40, 128, 136, active_ids_v3),
])
v1_dir = write_version("v1", 500, history)
v2_dir = write_version("v2", 620, pd.concat([history, week_2]))
v3_dir = write_version("v3", 690, pd.concat([history, week_2, week_3]))

## Build a graph and snapshot for each version

`load` builds the graph, query and snapshot for a folder. The query asks whether a customer transacts in the next seven days.
Refreshing means calling `load` on the updated files, which supplies new data without fitting new model weights.
See [Graphs](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) and [Predictive Queries](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries).

In [5]:
Version = namedtuple("Version", "folder connector query snapshot")


def load(folder):
    connector = kumo.FileConnector(root_dir=str(folder))
    customers = kumo.Table.from_source_table(
        connector["customers.parquet"], primary_key="CUSTOMER_ID"
    ).infer_metadata()
    transactions = kumo.Table.from_source_table(
        connector["transactions.parquet"], primary_key="TX_ID", time_column="TS"
    ).infer_metadata().with_column("CUSTOMER_ID", stype=Stype.ID)
    graph = kumo.Graph(
        tables={"customers": customers, "transactions": transactions},
        edges=[kumo.Edge(src_table="transactions", fkey="CUSTOMER_ID", dst_table="customers")],
    )
    graph.validate()
    query = kumo.PredictiveQuery(
        graph, "PREDICT COUNT(transactions.*, 0, 7, days) > 0 FOR EACH customers.CUSTOMER_ID"
    )
    return Version(folder, connector, query, graph.snapshot())


v1 = load(v1_dir)
print("Task type:", v1.query.get_task_type())

Task type: binary_classification


## Train once on `v1`

The FAST plan uses one experiment without refitting, and explanations are disabled, to keep this module on scoring rather than tuning.
See [Training](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training).

In [6]:
def fast_plan(query, snapshot, **suggest_options):
    model_plan = query.suggest_model_plan(snapshot, run_mode=RunMode.FAST, **suggest_options)
    model_plan.training_job.num_experiments = 1
    model_plan.training_job.refit_trainval = False
    model_plan.training_job.refit_full = False
    model_plan.training_job.disable_explain = True
    if SMOKE_TEST:
        model_plan.optimization.max_epochs = 2
        model_plan.optimization.max_steps_per_epoch = 20
    return model_plan


trainer = kumo.Trainer(model_plan=fast_plan(v1.query, v1.snapshot))
fit_result = trainer.fit(v1.query.generate_training_table(v1.snapshot), snapshot=v1.snapshot)

## Define one scoring run

`customers` has no created timestamp to filter on, so "new since last run" is the current customer IDs minus the IDs saved by the previous run.
Treat the saved IDs as job state: here they live in the workspace, and in production they belong next to your other job state. Start from the customers the model was trained on.

In [7]:
state_path = workspace / "last_run_ids.parquet"
pd.DataFrame({"CUSTOMER_ID": existing_ids}).to_parquet(state_path, index=False)

A run takes the refreshed version and does four things:

1. Take the anchor from the default prediction table of the refreshed snapshot, which is the latest time in the data, so the daily and weekly runs describe the same moment.
2. Select the new customers by subtracting the saved IDs from the current IDs.
3. Score the new customers (daily) and every customer (weekly) with the model trained on `v1`. Both are the same call with a different `ENTITY1` list.
4. Save the current IDs for the next run.

In [8]:
def score(version, name, entity_ids, anchor):
    pd.DataFrame({"ENTITY1": entity_ids, "TIMESTAMP": anchor}
                 ).to_parquet(version.folder / f"{name}.parquet", index=False)
    prediction_table = kumo.PredictionTable.from_source_table(
        version.connector[f"{name}.parquet"],
        pq=version.query,
        snapshot=version.snapshot,
        plan=trainer.training_table_generation_plan,
    )
    scores = trainer.predict(
        prediction_table, snapshot=version.snapshot, binary_classification_threshold=0.5
    ).frame
    return scores.set_index("ENTITY").sort_index()


def run(version, name):
    default_table = version.query.generate_prediction_table(
        version.snapshot, plan=trainer.training_table_generation_plan
    )
    anchor = pd.Timestamp(default_table.timestamps[0], unit="ns")
    last_run_ids = set(pd.read_parquet(state_path).CUSTOMER_ID)
    current_ids = set(pd.read_parquet(version.folder / "customers.parquet").CUSTOMER_ID)
    new_ids = sorted(current_ids - last_run_ids)

    daily = score(version, f"{name}_daily", new_ids, anchor)
    weekly = score(version, f"{name}_weekly", sorted(current_ids), anchor)
    pd.DataFrame({"CUSTOMER_ID": sorted(current_ids)}).to_parquet(state_path, index=False)
    print(f"Anchor {anchor} | {len(daily)} daily rows, {len(weekly)} weekly rows, "
          f"{daily.True_PROB.isna().sum() + weekly.True_PROB.isna().sum()} missing")
    return anchor, daily, weekly

## First refresh

Build the snapshot for `v2`. The two snapshots have different IDs; the trained model is unchanged.
Expect 120 daily rows, 620 weekly rows and no missing probabilities. Customers the model never saw, including those with no transactions, still receive scores.

In [9]:
v2 = load(v2_dir)
print("Training snapshot: ", v1.snapshot.id)
print("Refreshed snapshot:", v2.snapshot.id)
anchor_2, daily_2, weekly_2 = run(v2, "refresh_1")
daily_2.head(3)

Training snapshot:  54c873b61ea14f6171e471ef09ef11bbe0486bbd51d3b4d8a61fd0c0ee129643@67368334375e41ad
Refreshed snapshot: 5c748157d850240b771eeb11dbb1d77848c17ea1f7bfa212fae8de004ee1e308@67368334375e41ad


Anchor 2024-05-07 00:00:00 | 120 daily rows, 620 weekly rows, 0 missing


,TARGET_PRED,False_PROB,True_PROB
ENTITY,,,
500,False,0.734485,0.265515
501,False,0.703044,0.296956
502,False,0.735326,0.264674


## Second refresh

Repeat with the next version of the data. The saved IDs now include the customers from the first refresh, so only the 70 customers added since then are new.
Expect 70 daily rows and 690 weekly rows.

In [10]:
v3 = load(v3_dir)
anchor_3, daily_3, weekly_3 = run(v3, "refresh_2")
daily_3.head(3)

Anchor 2024-05-15 00:00:00 | 70 daily rows, 690 weekly rows, 0 missing


,TARGET_PRED,False_PROB,True_PROB
ENTITY,,,
620,False,0.735518,0.264482
621,False,0.735416,0.264584
622,False,0.703052,0.296948


## Compare the scores

- **Daily versus weekly.** For the same customer, snapshot and anchor, scores agree up to small numerical differences, so compare with a tolerance.
- **No activity.** New customers with no transactions are scored from their own columns alone, so there is no activity signal behind their scores.
- **Across snapshots.** Features come from the snapshot, so the same customer scores slightly differently on `v1` and `v2`. Compare scores only within one snapshot and anchor, or recompute older scores after a refresh.

These scores come from synthetic data and a small model: demonstration only.

In [11]:
on_v1 = score(v1, "existing_at_anchor", existing_ids, anchor_2)
inactive_ids_v2 = np.arange(600, 620)
print(f"Largest daily vs weekly difference, refresh 1: {(daily_2.True_PROB - weekly_2.True_PROB.loc[daily_2.index]).abs().max():.1e}")
print(f"Largest daily vs weekly difference, refresh 2: {(daily_3.True_PROB - weekly_3.True_PROB.loc[daily_3.index]).abs().max():.1e}")
print(f"Mean probability, new without activity (refresh 1): {weekly_2.True_PROB.loc[inactive_ids_v2].mean():.3f}")
print(f"Mean probability, existing customers (refresh 1):   {weekly_2.True_PROB.loc[existing_ids].mean():.3f}")
print(f"Mean difference, v1 vs v2 snapshot:                 {(on_v1.True_PROB - weekly_2.True_PROB.loc[existing_ids]).abs().mean():.4f}")

Largest daily vs weekly difference, refresh 1: 8.9e-07
Largest daily vs weekly difference, refresh 2: 1.8e-06
Mean probability, new without activity (refresh 1): 0.297
Mean probability, existing customers (refresh 1):   0.266
Mean difference, v1 vs v2 snapshot:                 0.0001


## Next step

Train once, then on each run rebuild the graph and snapshot from the updated files, subtract the saved IDs from the current IDs for the daily population, use every current ID for the weekly one and save the current IDs afterward.
To choose rows and anchor times yourself, see [custom prediction table](local_kumo_engine_custom_prediction_table.ipynb); for the static-query variant, see [static query predictions](local_kumo_engine_static_query_predictions.ipynb).
For scheduling, see [Schedule prediction jobs](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/deploy-a-model/schedule-prediction-jobs).

## Cleanup

Remove this notebook's workspace, which holds only the files it created. Skip this cell to keep the generated tables and predictions.

In [12]:
import shutil

shutil.rmtree(workspace)